최종 학습 데이터셋을 만드는 것이 아니라 다음을 확인하는 단계다.

계산 가능한가?
결측이나 이상치가 많은가?
이탈자와 유지자 사이에 차이가 있는가?
데이터 누수 위험은 없는가?
최종 모델 피처로 사용할 가치가 있는가?

피처 검증 순서
순서	피처 그룹	우선순위
1	리뷰 활동량 변화	P0
2	리뷰 작성 간격 변화	P0
3	신규 음식점 탐색 변화	P0
4	카테고리 다양성 변화	P0
5	탐방 반경 변화	P1
6	평점 성향 변화	P1
7	Useful·Cool 반응 변화	조건부

In [1]:
123

123

In [2]:
# 1. 라이브러리
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds
import yaml

In [3]:
# 2. 경로
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "data" / "interim").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "data/interim 폴더를 찾을 수 없습니다."
    )

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "analysis_config.yaml"
)

RESTAURANT_REVIEW_PATH = (
    INTERIM_DIR
    / "restaurant_reviews.parquet"
)

RESTAURANT_BUSINESS_PATH = (
    INTERIM_DIR
    / "restaurant_businesses.parquet"
)

print("설정 파일:", CONFIG_PATH.exists())
print("리뷰 파일:", RESTAURANT_REVIEW_PATH.exists())
print("음식점 파일:", RESTAURANT_BUSINESS_PATH.exists())

설정 파일: True
리뷰 파일: True
음식점 파일: True


In [4]:
# 3. 설정 파일 불러오기
with CONFIG_PATH.open(
    mode="r",
    encoding="utf-8"
) as file:
    config = yaml.safe_load(file)

cohort_version = config[
    "project"
]["cohort_version"]

power_config = config[
    "power_reviewer"
]

recent_config = config[
    "recent_activity"
]

target_config = config[
    "target"
]

In [5]:
FINAL_COHORT_PATH = (
    INTERIM_DIR
    / f"power_reviewer_cohort_{cohort_version}.parquet"
)

print(
    "최종 코호트 파일:",
    FINAL_COHORT_PATH.exists()
)

최종 코호트 파일: True


In [6]:
# 4. 최종 코호트 불러오기
cohort_df = pd.read_parquet(
    FINAL_COHORT_PATH
)

print("코호트 크기:", cohort_df.shape)
print(
    "고유 사용자:",
    cohort_df["user_id"].nunique()
)

cohort_df["churn"].value_counts()


코호트 크기: (3908, 9)
고유 사용자: 3908


churn
0    3302
1     606
Name: count, dtype: int64

In [7]:
# 5. 관찰 기간 리뷰만 추출
cohort_user_ids = (
    cohort_df["user_id"]
    .astype(str)
    .tolist()
)

review_dataset = ds.dataset(
    RESTAURANT_REVIEW_PATH,
    format="parquet"
)

In [8]:
observation_filter = (
    (
        ds.field("date")
        >= power_config["baseline_start"]
    )
    & (
        ds.field("date")
        < recent_config["observation_end"]
    )
    & (
        ds.field("user_id")
        .isin(cohort_user_ids)
    )
)

In [9]:
cohort_review_table = (
    review_dataset.to_table(
        filter=observation_filter,
        columns=[
            "review_id",
            "user_id",
            "business_id",
            "stars",
            "useful",
            "funny",
            "cool",
            "date"
        ]
    )
)

cohort_review_df = (
    cohort_review_table.to_pandas()
)

print(
    "관찰 기간 코호트 리뷰:",
    cohort_review_df.shape
)

관찰 기간 코호트 리뷰: (144482, 8)


In [10]:
# 6. 날짜 변환 및 기간 구분
cohort_review_df["date"] = pd.to_datetime(
    cohort_review_df["date"],
    errors="coerce"
)

cohort_review_df["period"] = np.where(
    cohort_review_df["date"]
    < pd.Timestamp(
        power_config["baseline_end"]
    ),
    "baseline",
    "recent"
)

In [11]:
observation_validation_df = pd.DataFrame(
    [
        {
            "review_rows": len(
                cohort_review_df
            ),
            "unique_users": (
                cohort_review_df[
                    "user_id"
                ].nunique()
            ),
            "date_null_rows": (
                cohort_review_df[
                    "date"
                ].isna().sum()
            ),
            "min_date": (
                cohort_review_df[
                    "date"
                ].min()
            ),
            "max_date": (
                cohort_review_df[
                    "date"
                ].max()
            )
        }
    ]
)

observation_validation_df

,review_rows,unique_users,date_null_rows,min_date,max_date
0,144482,3908,0,2017-01-01 00:26:05,2018-12-31 23:52:36


정상이라면:

고유 사용자 수가 3,908명
날짜 결측이 0건
최소 날짜가 2017년 1월 이후
최대 날짜가 2018년 12월 31일 이하여야 함

In [12]:
cohort_review_df.groupby(
    "period"
).size()

period
baseline    84375
recent      60107
dtype: int64

In [13]:
# 7. 음식점 정보 결합
restaurant_business_df = pd.read_parquet(
    RESTAURANT_BUSINESS_PATH,
    columns=[
        "business_id",
        "city",
        "state",
        "latitude",
        "longitude",
        "categories"
    ]
)

In [14]:
cohort_review_df = (
    cohort_review_df
    .merge(
        restaurant_business_df,
        on="business_id",
        how="left",
        validate="many_to_one"
    )
)

print(
    "결합 후 리뷰 수:",
    len(cohort_review_df)
)

결합 후 리뷰 수: 144482


In [15]:
business_merge_validation_df = pd.DataFrame(
    [
        {
            "total_rows": len(
                cohort_review_df
            ),
            "missing_business_rows": (
                cohort_review_df[
                    "categories"
                ].isna().sum()
            ),
            "missing_latitude_rows": (
                cohort_review_df[
                    "latitude"
                ].isna().sum()
            ),
            "missing_longitude_rows": (
                cohort_review_df[
                    "longitude"
                ].isna().sum()
            )
        }
    ]
)

business_merge_validation_df

,total_rows,missing_business_rows,missing_latitude_rows,missing_longitude_rows
0,144482,0,0,0


In [ ]:
# 8. 관찰 기간 데이터 저장

# 앞으로 피처를 만들 때 472만 건을 매번 다시 필터링하지 않도록 작은 코호트 데이터로 저장.

In [16]:
COHORT_OBSERVATION_PATH = (
    INTERIM_DIR
    / (
        "cohort_observation_reviews_"
        f"{cohort_version}.parquet"
    )
)

cohort_review_df.to_parquet(
    COHORT_OBSERVATION_PATH,
    index=False
)

print(
    "관찰 기간 데이터 저장:",
    COHORT_OBSERVATION_PATH
)

관찰 기간 데이터 저장: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\cohort_observation_reviews_v01.parquet


In [17]:
observation_validation_df

,review_rows,unique_users,date_null_rows,min_date,max_date
0,144482,3908,0,2017-01-01 00:26:05,2018-12-31 23:52:36


In [18]:
business_merge_validation_df

,total_rows,missing_business_rows,missing_latitude_rows,missing_longitude_rows
0,144482,0,0,0


검증 결과가 모두 정상.

코호트 사용자 3,908명 전원 포함
관찰 기간 리뷰 144,482건
날짜 변환 결측 0건
2017-01-01부터 2018-12-31까지 정상
Business 연결 결측 0건

In [19]:
# 9. 리뷰 연월 생성
cohort_review_df["review_month"] = (
    cohort_review_df["date"]
    .dt.to_period("M")
)

In [20]:
# 10. 사용자·기간별 활동량 집계
activity_period_df = (
    cohort_review_df
    .groupby(
        [
            "user_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        review_count=(
            "review_id",
            "count"
        ),
        active_months=(
            "review_month",
            "nunique"
        )
    )
)

activity_period_df.head()

,user_id,period,review_count,active_months
0,--Vu3Gux9nPnLcG9yO_HxA,baseline,18,5
1,--Vu3Gux9nPnLcG9yO_HxA,recent,4,4
2,--u09WAjW741FdfkJXxNmg,baseline,18,10
3,--u09WAjW741FdfkJXxNmg,recent,11,9
4,-0YrXUvXz8112yHap35V2g,baseline,10,5


In [22]:
# 11. 2017년 기준 활동 생성
baseline_activity_df = (
    activity_period_df[
        activity_period_df["period"]
        == "baseline"
    ][
        [
            "user_id",
            "review_count",
            "active_months"
        ]
    ]
    .rename(
        columns={
            "review_count": (
                "baseline_review_count"
            ),
            "active_months": (
                "baseline_active_months"
            )
        }
    )
)

baseline_activity_df.head()

,user_id,baseline_review_count,baseline_active_months
0,--Vu3Gux9nPnLcG9yO_HxA,18,5
2,--u09WAjW741FdfkJXxNmg,18,10
4,-0YrXUvXz8112yHap35V2g,10,5
6,-0rW5xrSCH1HCnSEmoQA5A,21,9
8,-1MF2tosrw2WcCxeVNk81Q,13,6


In [23]:
# 12. 2018년 최근 활동 생성
recent_activity_df = (
    activity_period_df[
        activity_period_df["period"]
        == "recent"
    ][
        [
            "user_id",
            "review_count",
            "active_months"
        ]
    ]
    .rename(
        columns={
            "review_count": (
                "recent_review_count"
            ),
            "active_months": (
                "recent_active_months"
            )
        }
    )
)

recent_activity_df.head()

,user_id,recent_review_count,recent_active_months
1,--Vu3Gux9nPnLcG9yO_HxA,4,4
3,--u09WAjW741FdfkJXxNmg,11,9
5,-0YrXUvXz8112yHap35V2g,3,2
7,-0rW5xrSCH1HCnSEmoQA5A,7,5
9,-1MF2tosrw2WcCxeVNk81Q,4,3


In [24]:
# 13. 활동 피처 결합
activity_feature_df = (
    baseline_activity_df
    .merge(
        recent_activity_df,
        on="user_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        cohort_df[
            [
                "user_id",
                "churn"
            ]
        ],
        on="user_id",
        how="inner",
        validate="one_to_one"
    )
)

print(
    "활동 피처 데이터 크기:",
    activity_feature_df.shape
)

activity_feature_df.head()

활동 피처 데이터 크기: (3908, 6)


,user_id,baseline_review_count,baseline_active_months,recent_review_count,recent_active_months,churn
0,--Vu3Gux9nPnLcG9yO_HxA,18,5,4,4,1
1,--u09WAjW741FdfkJXxNmg,18,10,11,9,0
2,-0YrXUvXz8112yHap35V2g,10,5,3,2,0
3,-0rW5xrSCH1HCnSEmoQA5A,21,9,7,5,0
4,-1MF2tosrw2WcCxeVNk81Q,13,6,4,3,0


In [25]:
# 14. 기존 코호트 집계값과 비교
activity_recalculation_check_df = (
    activity_feature_df
    .merge(
        cohort_df[
            [
                "user_id",
                "baseline_review_count",
                "change_review_count",
                "baseline_active_months",
                "change_active_months"
            ]
        ],
        on="user_id",
        how="left",
        suffixes=(
            "_recalculated",
            "_cohort"
        ),
        validate="one_to_one"
    )
)

In [26]:
assert (
    activity_recalculation_check_df[
        "baseline_review_count_recalculated"
    ]
    == activity_recalculation_check_df[
        "baseline_review_count_cohort"
    ]
).all()

assert (
    activity_recalculation_check_df[
        "recent_review_count"
    ]
    == activity_recalculation_check_df[
        "change_review_count"
    ]
).all()

assert (
    activity_recalculation_check_df[
        "baseline_active_months_recalculated"
    ]
    == activity_recalculation_check_df[
        "baseline_active_months_cohort"
    ]
).all()

assert (
    activity_recalculation_check_df[
        "recent_active_months"
    ]
    == activity_recalculation_check_df[
        "change_active_months"
    ]
).all()

print("활동량 재계산 검증 통과")

활동량 재계산 검증 통과


In [31]:
# 15. 리뷰 작성량 변화 피처
# 2018년 리뷰 수 - 2017년 리뷰 수
activity_feature_df[
    "review_count_diff"
] = (
    activity_feature_df[
        "recent_review_count"
    ]
    - activity_feature_df[
        "baseline_review_count"
    ]
)

In [28]:
# 2017년 대비 2018년 리뷰 작성량 비율
activity_feature_df[
    "review_count_ratio"
] = (
    activity_feature_df[
        "recent_review_count"
    ]
    / activity_feature_df[
        "baseline_review_count"
    ]
)

In [29]:
# 양수일수록 리뷰 작성량이 많이 감소함
activity_feature_df[
    "review_count_decline_rate"
] = (
    activity_feature_df[
        "baseline_review_count"
    ]
    - activity_feature_df[
        "recent_review_count"
    ]
) / activity_feature_df[
    "baseline_review_count"
]

In [32]:
# 16. 활동 월수 변화 피처
activity_feature_df[
    "active_month_diff"
] = (
    activity_feature_df[
        "recent_active_months"
    ]
    - activity_feature_df[
        "baseline_active_months"
    ]
)

In [33]:
activity_feature_df[
    "active_month_ratio"
] = (
    activity_feature_df[
        "recent_active_months"
    ]
    / activity_feature_df[
        "baseline_active_months"
    ]
)

In [34]:
# 양수일수록 활동한 월수가 많이 감소함
activity_feature_df[
    "active_month_decline_rate"
] = (
    activity_feature_df[
        "baseline_active_months"
    ]
    - activity_feature_df[
        "recent_active_months"
    ]
) / activity_feature_df[
    "baseline_active_months"
]

In [35]:
# 17. 활동 월당 리뷰 수
activity_feature_df[
    "baseline_reviews_per_active_month"
] = (
    activity_feature_df[
        "baseline_review_count"
    ]
    / activity_feature_df[
        "baseline_active_months"
    ]
)

In [36]:
activity_feature_df[
    "recent_reviews_per_active_month"
] = (
    activity_feature_df[
        "recent_review_count"
    ]
    / activity_feature_df[
        "recent_active_months"
    ]
)

In [37]:
activity_feature_df[
    "reviews_per_active_month_decline_rate"
] = (
    activity_feature_df[
        "baseline_reviews_per_active_month"
    ]
    - activity_feature_df[
        "recent_reviews_per_active_month"
    ]
) / activity_feature_df[
    "baseline_reviews_per_active_month"
]

In [38]:
# 18. 결측·무한대 검증
activity_numeric_columns = (
    activity_feature_df
    .select_dtypes(
        include="number"
    )
    .columns
)

activity_feature_validation_df = pd.DataFrame(
    {
        "feature": activity_numeric_columns,
        "missing_count": [
            activity_feature_df[column]
            .isna()
            .sum()
            for column in activity_numeric_columns
        ],
        "infinite_count": [
            np.isinf(
                activity_feature_df[column]
            ).sum()
            for column in activity_numeric_columns
        ]
    }
)

activity_feature_validation_df

,feature,missing_count,infinite_count
0,baseline_review_count,0,0
1,baseline_active_months,0,0
2,recent_review_count,0,0
3,recent_active_months,0,0
4,churn,0,0
5,review_count_diff,0,0
6,review_count_ratio,0,0
7,review_count_decline_rate,0,0
8,active_month_diff,0,0
9,active_month_ratio,0,0


In [39]:
# 19. 이탈자·유지자 비교
comparison_features = [
    "baseline_review_count",
    "recent_review_count",
    "review_count_decline_rate",
    "baseline_active_months",
    "recent_active_months",
    "active_month_decline_rate",
    "reviews_per_active_month_decline_rate"
]

In [40]:
activity_churn_summary_df = (
    activity_feature_df
    .groupby("churn")[
        comparison_features
    ]
    .agg(
        [
            "mean",
            "median"
        ]
    )
)

activity_churn_summary_df.columns = [
    f"{feature}_{stat}"
    for feature, stat
    in activity_churn_summary_df.columns
]

activity_churn_summary_df = (
    activity_churn_summary_df
    .reset_index()
)

activity_churn_summary_df

,churn,baseline_review_count_mean,baseline_review_count_median,recent_review_count_mean,recent_review_count_median,review_count_decline_rate_mean,review_count_decline_rate_median,baseline_active_months_mean,baseline_active_months_median,recent_active_months_mean,recent_active_months_median,active_month_decline_rate_mean,active_month_decline_rate_median,reviews_per_active_month_decline_rate_mean,reviews_per_active_month_decline_rate_median
0,0,22.220775,17.0,16.842217,13.0,0.184646,0.333333,7.216535,7.0,6.446699,6.0,0.053583,0.125000,0.157506,0.230769
1,1,18.155116,14.0,7.415842,5.0,0.556457,0.642857,5.998350,6.0,3.666667,3.0,0.344083,0.428571,0.330835,0.400000


In [41]:
# 20. 결과 저장
FEATURE_DIR = (
    INTERIM_DIR
    / "features"
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [42]:
activity_feature_df.to_parquet(
    FEATURE_DIR
    / f"activity_features_{cohort_version}.parquet",
    index=False
)

activity_feature_validation_df.to_csv(
    REPORT_TABLE_DIR
    / f"activity_feature_validation_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

activity_churn_summary_df.to_csv(
    REPORT_TABLE_DIR
    / f"activity_churn_summary_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("리뷰 활동량 피처 저장 완료")

리뷰 활동량 피처 저장 완료


In [43]:
# 리뷰 작성 간격 피처
# 1. 사용자별 리뷰 날짜 정렬
interval_source_df = (
    cohort_review_df[
        [
            "user_id",
            "period",
            "date"
        ]
    ]
    .sort_values(
        [
            "user_id",
            "period",
            "date"
        ]
    )
    .copy()
)

In [44]:
# 2. 이전 리뷰와의 간격 계산
interval_source_df[
    "interval_days"
] = (
    interval_source_df
    .groupby(
        [
            "user_id",
            "period"
        ]
    )["date"]
    .diff()
    .dt.total_seconds()
    / (60 * 60 * 24)
)

interval_source_df.head(20)

,user_id,period,date,interval_days
61295,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-03-20 14:56:25,NaN
17671,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-03-20 14:58:43,0.001597
138065,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-04-11 19:05:45,22.171551
102294,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-04-21 15:54:13,9.866991
34275,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-04-26 20:01:29,5.171713
52916,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-04-26 20:08:17,0.004722
105236,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-05-02 21:55:57,6.074769
59183,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-05-13 19:42:49,10.907546
30915,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-07-10 13:53:38,57.757512
132330,--Vu3Gux9nPnLcG9yO_HxA,baseline,2017-07-10 13:58:04,0.003079


In [45]:
# 3. 사용자·기간별 간격 집계
interval_period_df = (
    interval_source_df
    .groupby(
        [
            "user_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        period_review_count=(
            "date",
            "count"
        ),
        mean_interval_days=(
            "interval_days",
            "mean"
        ),
        median_interval_days=(
            "interval_days",
            "median"
        ),
        max_interval_days=(
            "interval_days",
            "max"
        ),
        first_review_date=(
            "date",
            "min"
        ),
        last_review_date=(
            "date",
            "max"
        )
    )
)

interval_period_df.head()

,user_id,period,period_review_count,mean_interval_days,median_interval_days,max_interval_days,first_review_date,last_review_date
0,--Vu3Gux9nPnLcG9yO_HxA,baseline,18,8.602178,5.015868,57.757512,2017-03-20 14:56:25,2017-08-13 20:37:44
1,--Vu3Gux9nPnLcG9yO_HxA,recent,4,69.701578,82.129896,108.570220,2018-02-23 15:50:49,2018-09-20 18:21:38
2,--u09WAjW741FdfkJXxNmg,baseline,18,18.459692,18.635787,41.611748,2017-01-08 23:41:44,2017-11-18 19:14:59
3,--u09WAjW741FdfkJXxNmg,recent,11,35.314190,29.104323,113.823252,2018-01-03 22:28:14,2018-12-23 01:52:34
4,-0YrXUvXz8112yHap35V2g,baseline,10,27.063758,9.463079,117.462685,2017-04-18 02:47:41,2017-12-17 16:33:59


In [47]:
# 4. 기준 기간과 최근 기간 분리
baseline_interval_df = (
    interval_period_df[
        interval_period_df["period"]
        == "baseline"
    ]
    .drop(columns="period")
    .rename(
        columns={
            "period_review_count": (
                "baseline_period_review_count"
            ),
            "mean_interval_days": (
                "baseline_mean_interval_days"
            ),
            "median_interval_days": (
                "baseline_median_interval_days"
            ),
            "max_interval_days": (
                "baseline_max_interval_days"
            ),
            "first_review_date": (
                "baseline_first_review_date"
            ),
            "last_review_date": (
                "baseline_last_review_date"
            )
        }
    )
)

In [ ]:
recent_interval_df = (
    interval_period_df[
        interval_period_df["period"]
        == "recent"
    ]
    .drop(columns="period")
    .rename(
        columns={
            "period_review_count": (
                "recent_period_review_count"
            ),
            "mean_interval_days": (
                "recent_mean_interval_days"
            ),
            "median_interval_days": (
                "recent_median_interval_days"
            ),
            "max_interval_days": (
                "recent_max_interval_days"
            ),
            "first_review_date": (
                "recent_first_review_date"
            ),
            "last_review_date": (
                "recent_last_review_date"
            )
        }
    )
)

In [49]:
# 5. 간격 피처 결합
interval_feature_df = (
    cohort_df[
        [
            "user_id",
            "churn"
        ]
    ]
    .merge(
        baseline_interval_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_interval_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

print(
    "간격 피처 데이터 크기:",
    interval_feature_df.shape
)

간격 피처 데이터 크기: (3908, 14)


In [50]:
# 6. 최근 간격 계산 가능 여부
# 2018년에 리뷰가 1건뿐이면 작성 간격을 계산할 수 없어. 이를 0으로 채우면 안 됨.
interval_feature_df[
    "recent_interval_available"
] = (
    interval_feature_df[
        "recent_period_review_count"
    ] >= 2
).astype("int8")

In [51]:
interval_availability_df = (
    interval_feature_df
    .groupby("churn")
    .agg(
        users=(
            "user_id",
            "count"
        ),
        interval_available_users=(
            "recent_interval_available",
            "sum"
        ),
        interval_available_rate=(
            "recent_interval_available",
            "mean"
        )
    )
    .reset_index()
)

interval_availability_df[
    "interval_available_rate"
] = (
    interval_availability_df[
        "interval_available_rate"
    ]
    * 100
).round(2)

interval_availability_df

,churn,users,interval_available_users,interval_available_rate
0,0,3302,3222,97.58
1,1,606,527,86.96


In [52]:
# 7. 리뷰 간격 변화 피처
interval_feature_df[
    "mean_interval_increase_days"
] = (
    interval_feature_df[
        "recent_mean_interval_days"
    ]
    - interval_feature_df[
        "baseline_mean_interval_days"
    ]
)


In [53]:
interval_feature_df[
    "median_interval_increase_days"
] = (
    interval_feature_df[
        "recent_median_interval_days"
    ]
    - interval_feature_df[
        "baseline_median_interval_days"
    ]
)

In [54]:
interval_feature_df[
    "max_interval_increase_days"
] = (
    interval_feature_df[
        "recent_max_interval_days"
    ]
    - interval_feature_df[
        "baseline_max_interval_days"
    ]
)

In [55]:
# 8. 기준 시점까지의 최근 공백 계산

# 2017년과 2018년 말 기준 마지막 리뷰 이후 경과일을 계산.
baseline_cutoff = pd.Timestamp(
    power_config["baseline_end"]
)

recent_cutoff = pd.Timestamp(
    recent_config["observation_end"]
)

In [56]:
interval_feature_df[
    "baseline_recency_days"
] = (
    baseline_cutoff
    - interval_feature_df[
        "baseline_last_review_date"
    ]
).dt.total_seconds() / (60 * 60 * 24)

In [57]:
interval_feature_df[
    "recent_recency_days"
] = (
    recent_cutoff
    - interval_feature_df[
        "recent_last_review_date"
    ]
).dt.total_seconds() / (60 * 60 * 24)

In [58]:
interval_feature_df[
    "recency_increase_days"
] = (
    interval_feature_df[
        "recent_recency_days"
    ]
    - interval_feature_df[
        "baseline_recency_days"
    ]
)

In [59]:
# 9. 결측과 무한대 확인
interval_numeric_columns = [
    "baseline_mean_interval_days",
    "recent_mean_interval_days",
    "mean_interval_increase_days",
    "baseline_median_interval_days",
    "recent_median_interval_days",
    "median_interval_increase_days",
    "baseline_max_interval_days",
    "recent_max_interval_days",
    "max_interval_increase_days",
    "baseline_recency_days",
    "recent_recency_days",
    "recency_increase_days",
    "recent_interval_available"
]

In [60]:
interval_feature_validation_df = pd.DataFrame(
    {
        "feature": interval_numeric_columns,
        "missing_count": [
            interval_feature_df[column]
            .isna()
            .sum()
            for column in interval_numeric_columns
        ],
        "infinite_count": [
            np.isinf(
                interval_feature_df[column]
            ).sum()
            for column in interval_numeric_columns
        ]
    }
)

interval_feature_validation_df

,feature,missing_count,infinite_count
0,baseline_mean_interval_days,0,0
1,recent_mean_interval_days,159,0
2,mean_interval_increase_days,159,0
3,baseline_median_interval_days,0,0
4,recent_median_interval_days,159,0
5,median_interval_increase_days,159,0
6,baseline_max_interval_days,0,0
7,recent_max_interval_days,159,0
8,max_interval_increase_days,159,0
9,baseline_recency_days,0,0


In [61]:
recent_interval_unavailable_count = (
    interval_feature_df[
        "recent_interval_available"
    ] == 0
).sum()

recent_mean_interval_missing_count = (
    interval_feature_df[
        "recent_mean_interval_days"
    ].isna().sum()
)

print(
    "간격 계산 불가능 사용자:",
    recent_interval_unavailable_count
)

print(
    "최근 평균 간격 결측:",
    recent_mean_interval_missing_count
)

assert (
    recent_interval_unavailable_count
    == recent_mean_interval_missing_count
)

print("리뷰 간격 결측 원인 검증 통과")

간격 계산 불가능 사용자: 159
최근 평균 간격 결측: 159
리뷰 간격 결측 원인 검증 통과


In [62]:
# 10. 이탈자·유지자 비교
interval_comparison_features = [
    "baseline_mean_interval_days",
    "recent_mean_interval_days",
    "mean_interval_increase_days",
    "recent_recency_days",
    "recency_increase_days",
    "recent_interval_available"
]

In [63]:
interval_churn_summary_df = (
    interval_feature_df
    .groupby("churn")[
        interval_comparison_features
    ]
    .agg(
        [
            "mean",
            "median"
        ]
    )
)

interval_churn_summary_df.columns = [
    f"{feature}_{stat}"
    for feature, stat
    in interval_churn_summary_df.columns
]

interval_churn_summary_df = (
    interval_churn_summary_df
    .reset_index()
)

interval_churn_summary_df

,churn,baseline_mean_interval_days_mean,baseline_mean_interval_days_median,recent_mean_interval_days_mean,recent_mean_interval_days_median,mean_interval_increase_days_mean,mean_interval_increase_days_median,recent_recency_days_mean,recent_recency_days_median,recency_increase_days_mean,recency_increase_days_median,recent_interval_available_mean,recent_interval_available_median
0,0,16.915888,15.900127,31.597013,22.449987,14.771443,6.755001,46.050149,29.211852,10.932944,6.933287,0.975772,1.0
1,1,17.552510,16.731368,47.591190,34.809562,30.003849,16.082790,93.027180,93.213466,40.629375,39.658623,0.869637,1.0


In [64]:
# 11. 결과 저장
interval_feature_df.to_parquet(
    FEATURE_DIR
    / f"interval_features_{cohort_version}.parquet",
    index=False
)

interval_feature_validation_df.to_csv(
    REPORT_TABLE_DIR
    / f"interval_feature_validation_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

interval_churn_summary_df.to_csv(
    REPORT_TABLE_DIR
    / f"interval_churn_summary_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

interval_availability_df.to_csv(
    REPORT_TABLE_DIR
    / f"interval_availability_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("리뷰 작성 간격 피처 저장 완료")


리뷰 작성 간격 피처 저장 완료


In [65]:
#  다음 피처: 신규 음식점 탐색 가능성 검증
# 여기서는 먼저 Yelp 리뷰 데이터로 동일 음식점에 여러 번 리뷰한 기록이 충분한지 확인해야 해.

# 실제 방문 데이터가 아니므로 표현은 다음처럼 해야 한다.

# 신규 음식점 방문 ❌
# 새롭게 리뷰한 음식점 ✅
# 신규 리뷰 음식점 ✅
# 음식점 탐색 범위 ✅
# 1. 사용자×음식점 중복 확인
user_business_review_count_df = (
    cohort_review_df
    .groupby(
        [
            "user_id",
            "business_id"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "review_count"
        }
    )
)

In [94]:
duplicate_pair_summary_df = pd.DataFrame(
    [
        {
            "total_reviews": (
                total_review_count
            ),
            "unique_user_business_pairs": (
                unique_pair_count
            ),
            "duplicate_review_rows": (
                duplicate_review_count
            ),
            "repeated_pairs": (
                repeated_pair_count
            ),
            "repeated_pair_rate_pct": round(
                repeated_pair_count
                / unique_pair_count
                * 100,
                2
            ),
            "max_reviews_same_business": (
                maximum_review_count
            )
        }
    ]
)

duplicate_pair_summary_df

,total_reviews,unique_user_business_pairs,duplicate_review_rows,repeated_pairs,repeated_pair_rate_pct,max_reviews_same_business
0,144482,138203,6279,5477,3.96,18


In [95]:
# 2. 기간별 고유 음식점 수
period_business_df = (
    cohort_review_df[
        [
            "user_id",
            "period",
            "business_id"
        ]
    ]
    .drop_duplicates()
)


In [96]:
unique_business_period_df = (
    period_business_df
    .groupby(
        [
            "user_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        unique_business_count=(
            "business_id",
            "nunique"
        )
    )
)

In [97]:
baseline_business_df = (
    unique_business_period_df[
        unique_business_period_df["period"]
        == "baseline"
    ][
        [
            "user_id",
            "unique_business_count"
        ]
    ]
    .rename(
        columns={
            "unique_business_count":
                "baseline_unique_business_count"
        }
    )
)

In [98]:
recent_business_df = (
    unique_business_period_df[
        unique_business_period_df["period"]
        == "recent"
    ][
        [
            "user_id",
            "unique_business_count"
        ]
    ]
    .rename(
        columns={
            "unique_business_count":
                "recent_unique_business_count"
        }
    )
)

In [99]:
# 3. 2017년 음식점을 2018년에 다시 리뷰했는지 확인
baseline_pairs_df = (
    period_business_df[
        period_business_df["period"]
        == "baseline"
    ][
        [
            "user_id",
            "business_id"
        ]
    ]
    .drop_duplicates()
)

In [100]:
recent_pairs_df = (
    period_business_df[
        period_business_df["period"]
        == "recent"
    ][
        [
            "user_id",
            "business_id"
        ]
    ]
    .drop_duplicates()
)

In [101]:
# 두 기간에 모두 등장한 사용자×음식점 조합을 찾는다.
revisited_pairs_df = (
    recent_pairs_df
    .merge(
        baseline_pairs_df,
        on=[
            "user_id",
            "business_id"
        ],
        how="inner",
        validate="one_to_one"
    )
)

In [102]:
revisited_count_df = (
    revisited_pairs_df
    .groupby(
        "user_id",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "recent_revisited_business_count"
        }
    )
)

In [103]:
# 4. 음식점 탐색 피처 생성
exploration_feature_df = (
    cohort_df[
        [
            "user_id",
            "churn"
        ]
    ]
    .merge(
        baseline_business_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_business_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        revisited_count_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)


In [104]:
# 재리뷰 음식점이 없는 경우는 0으로 처리.
exploration_feature_df[
    "recent_revisited_business_count"
] = (
    exploration_feature_df[
        "recent_revisited_business_count"
    ]
    .fillna(0)
    .astype(int)
)

In [105]:
# 5. 신규 리뷰 음식점 수와 비율

# 여기서 ‘신규’는 정확히 말하면 2017년에 리뷰하지 않았지만 2018년에 리뷰한 음식점.

exploration_feature_df[
    "recent_new_vs_baseline_count"
] = (
    exploration_feature_df[
        "recent_unique_business_count"
    ]
    - exploration_feature_df[
        "recent_revisited_business_count"
    ]
)

In [106]:
exploration_feature_df[
    "recent_new_vs_baseline_rate"
] = (
    exploration_feature_df[
        "recent_new_vs_baseline_count"
    ]
    / exploration_feature_df[
        "recent_unique_business_count"
    ]
)

In [107]:
exploration_feature_df[
    "recent_revisit_rate"
] = (
    exploration_feature_df[
        "recent_revisited_business_count"
    ]
    / exploration_feature_df[
        "recent_unique_business_count"
    ]
)

In [108]:
# 6. 음식점 탐색 범위 변화
exploration_feature_df[
    "unique_business_diff"
] = (
    exploration_feature_df[
        "recent_unique_business_count"
    ]
    - exploration_feature_df[
        "baseline_unique_business_count"
    ]
)

In [109]:
exploration_feature_df[
    "unique_business_decline_rate"
] = (
    exploration_feature_df[
        "baseline_unique_business_count"
    ]
    - exploration_feature_df[
        "recent_unique_business_count"
    ]
) / exploration_feature_df[
    "baseline_unique_business_count"
]

In [110]:
# 7. 결측·무한대 검증
exploration_numeric_columns = (
    exploration_feature_df
    .select_dtypes(
        include="number"
    )
    .columns
)

In [111]:
exploration_validation_df = pd.DataFrame(
    {
        "feature": (
            exploration_numeric_columns
        ),
        "missing_count": [
            exploration_feature_df[column]
            .isna()
            .sum()
            for column
            in exploration_numeric_columns
        ],
        "infinite_count": [
            np.isinf(
                exploration_feature_df[column]
            ).sum()
            for column
            in exploration_numeric_columns
        ]
    }
)

exploration_validation_df


,feature,missing_count,infinite_count
0,churn,0,0
1,baseline_unique_business_count,0,0
2,recent_unique_business_count,0,0
3,recent_revisited_business_count,0,0
4,recent_new_vs_baseline_count,0,0
5,recent_new_vs_baseline_rate,0,0
6,recent_revisit_rate,0,0
7,unique_business_diff,0,0
8,unique_business_decline_rate,0,0


In [112]:
# 8. 이탈자·유지자 비교
exploration_comparison_features = [
    "baseline_unique_business_count",
    "recent_unique_business_count",
    "unique_business_decline_rate",
    "recent_new_vs_baseline_count",
    "recent_new_vs_baseline_rate",
    "recent_revisited_business_count",
    "recent_revisit_rate"
]

In [113]:
exploration_churn_summary_df = (
    exploration_feature_df
    .groupby("churn")[
        exploration_comparison_features
    ]
    .agg(
        [
            "mean",
            "median"
        ]
    )
)

exploration_churn_summary_df.columns = [
    f"{feature}_{stat}"
    for feature, stat
    in exploration_churn_summary_df.columns
]

exploration_churn_summary_df = (
    exploration_churn_summary_df
    .reset_index()
)

exploration_churn_summary_df

,churn,baseline_unique_business_count_mean,baseline_unique_business_count_median,recent_unique_business_count_mean,recent_unique_business_count_median,unique_business_decline_rate_mean,unique_business_decline_rate_median,recent_new_vs_baseline_count_mean,recent_new_vs_baseline_count_median,recent_new_vs_baseline_rate_mean,recent_new_vs_baseline_rate_median,recent_revisited_business_count_mean,recent_revisited_business_count_median,recent_revisit_rate_mean,recent_revisit_rate_median
0,0,21.507874,17.0,16.338886,12.0,0.184414,0.333333,15.798910,12.0,0.964487,1.0,0.539976,0.0,0.035513,0.0
1,1,17.722772,14.0,7.298680,5.0,0.553541,0.636364,7.056106,5.0,0.961036,1.0,0.242574,0.0,0.038964,0.0


In [114]:
# 9. 결과 저장
exploration_feature_df.to_parquet(
    FEATURE_DIR
    / f"exploration_features_{cohort_version}.parquet",
    index=False
)

duplicate_pair_summary_df.to_csv(
    REPORT_TABLE_DIR
    / f"user_business_duplicate_summary_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

exploration_validation_df.to_csv(
    REPORT_TABLE_DIR
    / f"exploration_feature_validation_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

exploration_churn_summary_df.to_csv(
    REPORT_TABLE_DIR
    / f"exploration_churn_summary_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("음식점 탐색 피처 저장 완료")


음식점 탐색 피처 저장 완료


지금 결과에는 서로 모순되는 값이 하나 있어.

중복 리뷰 행: 6,279건
2회 이상 리뷰한 사용자×음식점 조합: 5,477개
동일 음식점 최대 리뷰 수: 1건 ← 불가능

반복 조합이 5,477개라면 최대 리뷰 수는 최소 2 이상이어야 해. 따라서 해석하기 전에 집계 코드를 다시 확인해야 한다.

In [89]:
user_business_review_count_df[
    "review_count"
].describe()

count    138203.000000
mean          1.045433
std           0.251588
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          18.000000
Name: review_count, dtype: float64

In [90]:
user_business_review_count_df[
    "review_count"
].value_counts().sort_index()

review_count
1     132726
2       4900
3        457
4         78
5         21
6          9
7          5
8          4
12         1
17         1
18         1
Name: count, dtype: int64

In [91]:
user_business_review_count_df.sort_values(
    "review_count",
    ascending=False
).head(10)

,user_id,business_id,review_count
83690,a0DnfD31lNdiBTY2-YBBFA,C6YaSrMAzy3jJqinlFVudw,18
133748,xWmYN57XXZbg0LOK8WbbFQ,gfLsBY-xsNE9-ktiTvTvGA,17
100824,i-kVxYDUksuuuxGF1qVQxw,AbPQR7DfUZOwoUdgSPQSjw,12
83692,a0DnfD31lNdiBTY2-YBBFA,FCCnOgjjvwQJBAy3u2fcLA,8
18163,6kJFLAHV-tNsBEZaRTqEWQ,v7KzbLoHo9zC2xRpWEEp0A,8
18160,6kJFLAHV-tNsBEZaRTqEWQ,sdfj6S0qzV5bx3FMDJnGIg,8
110009,mVtmlFPxA8cBzhaumweIIA,aYHjJJ64whQsxZjx5rUe_w,8
36325,F3Zt68LUvs_VigdYEWrInA,QtgubTRRFHtEL_fmsMQRCg,7
74028,WVqS85AUR20gbSFkuKH8Ig,zwe9H6Xxqb1_E09A20Ptgg,7
133757,xWmYN57XXZbg0LOK8WbbFQ,x3pTbsABLypwEoTdwQOmSg,7


In [92]:
total_review_count = (
    user_business_review_count_df[
        "review_count"
    ].sum()
)

unique_pair_count = len(
    user_business_review_count_df
)

duplicate_review_count = (
    (
        user_business_review_count_df[
            "review_count"
        ] - 1
    )
    .clip(lower=0)
    .sum()
)

repeated_pair_count = (
    user_business_review_count_df[
        "review_count"
    ] >= 2
).sum()

maximum_review_count = (
    user_business_review_count_df[
        "review_count"
    ].max()
)

print("전체 리뷰:", total_review_count)
print("고유 사용자×음식점:", unique_pair_count)
print("중복 리뷰 행:", duplicate_review_count)
print("반복 조합:", repeated_pair_count)
print("동일 음식점 최대 리뷰:", maximum_review_count)

전체 리뷰: 144482
고유 사용자×음식점: 138203
중복 리뷰 행: 6279
반복 조합: 5477
동일 음식점 최대 리뷰: 18


In [93]:
assert total_review_count == len(
    cohort_review_df
)

assert duplicate_review_count == (
    len(cohort_review_df)
    - unique_pair_count
)

if repeated_pair_count > 0:
    assert maximum_review_count >= 2

print("사용자×음식점 집계 검증 통과")

사용자×음식점 집계 검증 통과


In [115]:
# 리뷰 활동량과 음식점 탐색 피처 결합
exploration_activity_check_df = (
    exploration_feature_df
    .merge(
        activity_feature_df[
            [
                "user_id",
                "baseline_review_count",
                "recent_review_count",
                "review_count_decline_rate"
            ]
        ],
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

In [116]:
# 리뷰 수와 고유 음식점 수의 상관관계
exploration_correlation_df = (
    exploration_activity_check_df[
        [
            "baseline_review_count",
            "baseline_unique_business_count",
            "recent_review_count",
            "recent_unique_business_count",
            "review_count_decline_rate",
            "unique_business_decline_rate"
        ]
    ]
    .corr()
)

exploration_correlation_df

,baseline_review_count,baseline_unique_business_count,recent_review_count,recent_unique_business_count,review_count_decline_rate,unique_business_decline_rate
baseline_review_count,1.000000,0.996085,0.585208,0.595812,0.095701,0.094681
baseline_unique_business_count,0.996085,1.000000,0.584378,0.597889,0.094299,0.096467
recent_review_count,0.585208,0.584378,1.000000,0.995523,-0.632530,-0.631495
recent_unique_business_count,0.595812,0.597889,0.995523,1.000000,-0.614412,-0.619959
review_count_decline_rate,0.095701,0.094299,-0.632530,-0.614412,1.000000,0.991502
unique_business_decline_rate,0.094681,0.096467,-0.631495,-0.619959,0.991502,1.000000


In [117]:
print(
    "2017년 리뷰 수·고유 음식점 수 상관계수:",
    exploration_correlation_df.loc[
        "baseline_review_count",
        "baseline_unique_business_count"
    ]
)

print(
    "2018년 리뷰 수·고유 음식점 수 상관계수:",
    exploration_correlation_df.loc[
        "recent_review_count",
        "recent_unique_business_count"
    ]
)

print(
    "리뷰 감소율·고유 음식점 감소율 상관계수:",
    exploration_correlation_df.loc[
        "review_count_decline_rate",
        "unique_business_decline_rate"
    ]
)

2017년 리뷰 수·고유 음식점 수 상관계수: 0.9960846846962916
2018년 리뷰 수·고유 음식점 수 상관계수: 0.9955227642900345
리뷰 감소율·고유 음식점 감소율 상관계수: 0.9915024012987207


In [118]:
# 5. 재리뷰 경험 사용자 비율
exploration_feature_df[
    "has_recent_revisit"
] = (
    exploration_feature_df[
        "recent_revisited_business_count"
    ] >= 1
).astype("int8")

In [119]:
revisit_user_summary_df = (
    exploration_feature_df
    .groupby("churn")
    .agg(
        users=(
            "user_id",
            "count"
        ),
        revisit_users=(
            "has_recent_revisit",
            "sum"
        ),
        revisit_user_rate=(
            "has_recent_revisit",
            "mean"
        )
    )
    .reset_index()
)

revisit_user_summary_df[
    "revisit_user_rate"
] = (
    revisit_user_summary_df[
        "revisit_user_rate"
    ]
    * 100
).round(2)

revisit_user_summary_df

,churn,users,revisit_users,revisit_user_rate
0,0,3302,995,30.13
1,1,606,112,18.48


In [120]:
# 6. 신규 리뷰 음식점 비율의 분산 확인
exploration_feature_df[
    "recent_new_vs_baseline_rate"
].describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

count    3908.000000
mean        0.963952
std         0.085804
min         0.000000
25%         0.964286
50%         1.000000
75%         1.000000
90%         1.000000
95%         1.000000
99%         1.000000
max         1.000000
Name: recent_new_vs_baseline_rate, dtype: float64

In [121]:
all_new_business_rate = (
    exploration_feature_df[
        "recent_new_vs_baseline_rate"
    ] == 1
).mean() * 100

print(
    f"2018년 리뷰 음식점이 모두 새로운 사용자: "
    f"{all_new_business_rate:.2f}%"
)

2018년 리뷰 음식점이 모두 새로운 사용자: 71.67%


In [122]:
# 전체 과거 이력 기준 신규 음식점 검증
# 1. 코호트 사용자의 2018년까지 전체 리뷰 추출
history_filter = (
    (
        ds.field("date")
        < recent_config["observation_end"]
    )
    & (
        ds.field("user_id")
        .isin(cohort_user_ids)
    )
)

In [123]:
cohort_history_table = (
    review_dataset.to_table(
        filter=history_filter,
        columns=[
            "user_id",
            "business_id",
            "date"
        ]
    )
)

cohort_history_df = (
    cohort_history_table.to_pandas()
)

cohort_history_df["date"] = (
    pd.to_datetime(
        cohort_history_df["date"],
        errors="coerce"
    )
)

print(
    "코호트 전체 과거 리뷰:",
    cohort_history_df.shape
)

print(
    "과거 리뷰 기간:",
    cohort_history_df["date"].min(),
    "~",
    cohort_history_df["date"].max()
)

코호트 전체 과거 리뷰: (300988, 3)
과거 리뷰 기간: 2006-07-18 20:52:38 ~ 2018-12-31 23:52:36


In [125]:
# 2. 사용자별 음식점 최초 리뷰일 계산
first_business_review_df = (
    cohort_history_df
    .groupby(
        [
            "user_id",
            "business_id"
        ],
        as_index=False
    )
    .agg(
        first_review_date=(
            "date",
            "min"
        )
    )
)

first_business_review_df.head()

,user_id,business_id,first_review_date
0,--Vu3Gux9nPnLcG9yO_HxA,8wktEb-euuXOi4fq1K4uGQ,2017-04-26 20:08:17
1,--Vu3Gux9nPnLcG9yO_HxA,CRrKt_OSlTXhOQ4VhONMkg,2017-07-10 13:58:04
2,--Vu3Gux9nPnLcG9yO_HxA,EYs_T4NU2BbaUF7vUDI55Q,2017-07-10 13:53:38
3,--Vu3Gux9nPnLcG9yO_HxA,IDjYyBrPW5C2CT9mXsyuBA,2017-03-20 14:58:43
4,--Vu3Gux9nPnLcG9yO_HxA,I_mCFePUG2MGuH-zuws7bA,2017-07-10 14:16:38


In [126]:
# 3. 2017년 신규 음식점 수
baseline_new_business_df = (
    first_business_review_df[
        (
            first_business_review_df[
                "first_review_date"
            ] >= power_config[
                "baseline_start"
            ]
        )
        & (
            first_business_review_df[
                "first_review_date"
            ] < power_config[
                "baseline_end"
            ]
        )
    ]
    .groupby(
        "user_id",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "baseline_new_business_count"
        }
    )
)

In [127]:
# 4. 2018년 신규 음식점 수
recent_new_business_df = (
    first_business_review_df[
        (
            first_business_review_df[
                "first_review_date"
            ] >= recent_config[
                "observation_start"
            ]
        )
        & (
            first_business_review_df[
                "first_review_date"
            ] < recent_config[
                "observation_end"
            ]
        )
    ]
    .groupby(
        "user_id",
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size":
                "recent_new_business_count"
        }
    )
)

In [128]:
# 5. 기존 탐색 피처와 결합
historical_exploration_df = (
    exploration_feature_df[
        [
            "user_id",
            "churn",
            "baseline_unique_business_count",
            "recent_unique_business_count"
        ]
    ]
    .merge(
        baseline_new_business_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_new_business_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

신규 음식점이 없는 사용자는 0으로 처리.

In [129]:
new_count_columns = [
    "baseline_new_business_count",
    "recent_new_business_count"
]

historical_exploration_df[
    new_count_columns
] = (
    historical_exploration_df[
        new_count_columns
    ]
    .fillna(0)
    .astype(int)
)

In [130]:
# 6. 진짜 신규 음식점 비율 계산
historical_exploration_df[
    "baseline_new_business_rate"
] = (
    historical_exploration_df[
        "baseline_new_business_count"
    ]
    / historical_exploration_df[
        "baseline_unique_business_count"
    ]
)

In [131]:
historical_exploration_df[
    "recent_new_business_rate"
] = (
    historical_exploration_df[
        "recent_new_business_count"
    ]
    / historical_exploration_df[
        "recent_unique_business_count"
    ]
)

In [132]:
# 양수일수록 신규 음식점 탐색 비율이 감소
historical_exploration_df[
    "new_business_rate_decline"
] = (
    historical_exploration_df[
        "baseline_new_business_rate"
    ]
    - historical_exploration_df[
        "recent_new_business_rate"
    ]
)

In [133]:
# 신규 음식점 수 변화
historical_exploration_df[
    "new_business_count_diff"
] = (
    historical_exploration_df[
        "recent_new_business_count"
    ]
    - historical_exploration_df[
        "baseline_new_business_count"
    ]
)

In [134]:
# 7. 이탈자·유지자 비교
historical_exploration_summary_df = (
    historical_exploration_df
    .groupby("churn")[
        [
            "baseline_new_business_count",
            "recent_new_business_count",
            "baseline_new_business_rate",
            "recent_new_business_rate",
            "new_business_rate_decline",
            "new_business_count_diff"
        ]
    ]
    .agg(
        [
            "mean",
            "median"
        ]
    )
)

historical_exploration_summary_df.columns = [
    f"{feature}_{stat}"
    for feature, stat
    in historical_exploration_summary_df.columns
]

historical_exploration_summary_df = (
    historical_exploration_summary_df
    .reset_index()
)

historical_exploration_summary_df

,churn,baseline_new_business_count_mean,baseline_new_business_count_median,recent_new_business_count_mean,recent_new_business_count_median,baseline_new_business_rate_mean,baseline_new_business_rate_median,recent_new_business_rate_mean,recent_new_business_rate_median,new_business_rate_decline_mean,new_business_rate_decline_median,new_business_count_diff_mean,new_business_count_diff_median
0,0,20.669897,16.0,15.389461,12.0,0.960965,1.0,0.939552,1.0,0.021414,0.0,-5.280436,-6.0
1,1,17.382838,14.0,6.963696,5.0,0.978563,1.0,0.948377,1.0,0.030185,0.0,-10.419142,-9.0


In [135]:
# 8. 신규 음식점 비율 분포 확인
historical_exploration_df[
    [
        "baseline_new_business_rate",
        "recent_new_business_rate",
        "new_business_rate_decline"
    ]
].describe(
    percentiles=[
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)

,baseline_new_business_rate,recent_new_business_rate,new_business_rate_decline
count,3908.000000,3908.000000,3908.000000
mean,0.963694,0.940920,0.022774
std,0.070160,0.111771,0.109807
min,0.402985,0.000000,-0.523810
25%,0.944444,0.916667,0.000000
50%,1.000000,1.000000,0.000000
75%,1.000000,1.000000,0.045455
90%,1.000000,1.000000,0.142857
95%,1.000000,1.000000,0.201579
99%,1.000000,1.000000,0.416352


In [136]:
# 9. 결과 저장
historical_exploration_df.to_parquet(
    FEATURE_DIR
    / (
        "historical_exploration_features_"
        f"{cohort_version}.parquet"
    ),
    index=False
)

historical_exploration_summary_df.to_csv(
    REPORT_TABLE_DIR
    / (
        "historical_exploration_summary_"
        f"{cohort_version}.csv"
    ),
    index=False,
    encoding="utf-8-sig"
)

In [137]:
# 다음 피처: 음식점 카테고리 다양성

# 카테고리 수 역시 리뷰 수의 영향을 받을 수 있으므로 다음 두 종류를 같이 확인해야 해.

# 고유 카테고리 수: 탐색한 카테고리의 절대 개수
# 정규화 엔트로피: 리뷰 수와 카테고리 수의 영향을 일부 보정한 다양성
# 1. 카테고리 데이터 생성
# 음식점 한 곳에 리뷰가 여러 번 있더라도 카테고리를 반복 집계하지 않도록 사용자·기간·음식점 기준으로 중복을 제거
category_source_df = (
    cohort_review_df[
        [
            "user_id",
            "period",
            "business_id",
            "categories"
        ]
    ]
    .drop_duplicates(
        subset=[
            "user_id",
            "period",
            "business_id"
        ]
    )
    .copy()
)

In [138]:
category_source_df["category"] = (
    category_source_df["categories"]
    .str.split(", ")
)

category_exploded_df = (
    category_source_df
    .explode("category")
)

In [139]:
# 2. 일반 카테고리 제외

# 모든 업체에 공통으로 붙는 상위 카테고리는 다양성 계산에서 제외
excluded_categories = {
    "Restaurants",
    "Food",
    "Nightlife",
    "Event Planning & Services"
}

In [140]:
category_exploded_df = (
    category_exploded_df[
        ~category_exploded_df[
            "category"
        ].isin(excluded_categories)
    ]
    .dropna(
        subset=["category"]
    )
    .copy()
)

print(
    "카테고리 전개 행 수:",
    len(category_exploded_df)
)

print(
    "고유 카테고리 수:",
    category_exploded_df[
        "category"
    ].nunique()
)

카테고리 전개 행 수: 555035
고유 카테고리 수: 586


In [141]:
# 3. 사용자·기간·카테고리별 빈도
category_frequency_df = (
    category_exploded_df
    .groupby(
        [
            "user_id",
            "period",
            "category"
        ],
        as_index=False
    )
    .size()
    .rename(
        columns={
            "size": "category_count"
        }
    )
)

category_frequency_df.head()

,user_id,period,category,category_count
0,--Vu3Gux9nPnLcG9yO_HxA,baseline,American (New),4
1,--Vu3Gux9nPnLcG9yO_HxA,baseline,American (Traditional),3
2,--Vu3Gux9nPnLcG9yO_HxA,baseline,Bagels,1
3,--Vu3Gux9nPnLcG9yO_HxA,baseline,Bakeries,2
4,--Vu3Gux9nPnLcG9yO_HxA,baseline,Barbeque,1


In [142]:
# 4. 다양성 계산 함수
def calculate_category_diversity(group):
    """
    사용자별 카테고리 분포의 다양성을 계산한다.
    """
    counts = (
        group["category_count"]
        .to_numpy(
            dtype=float
        )
    )

    probabilities = (
        counts
        / counts.sum()
    )

    unique_category_count = len(
        counts
    )

    entropy = -np.sum(
        probabilities
        * np.log(probabilities)
    )

    if unique_category_count > 1:
        normalized_entropy = (
            entropy
            / np.log(
                unique_category_count
            )
        )
    else:
        normalized_entropy = 0.0

    simpson_diversity = (
        1
        - np.sum(
            probabilities ** 2
        )
    )

    top_category_share = (
        probabilities.max()
    )

    return pd.Series(
        {
            "unique_category_count":
                unique_category_count,
            "normalized_category_entropy":
                normalized_entropy,
            "simpson_category_diversity":
                simpson_diversity,
            "top_category_share":
                top_category_share
        }
    )

In [143]:
# 5. 사용자·기간별 다양성 계산
category_period_feature_df = (
    category_frequency_df
    .groupby(
        [
            "user_id",
            "period"
        ]
    )
    .apply(
        calculate_category_diversity,
        include_groups=False
    )
    .reset_index()
)

category_period_feature_df.head()

,user_id,period,unique_category_count,normalized_category_entropy,simpson_category_diversity,top_category_share
0,--Vu3Gux9nPnLcG9yO_HxA,baseline,36.0,0.940290,0.958045,0.088235
1,--Vu3Gux9nPnLcG9yO_HxA,recent,13.0,0.990287,0.918367,0.142857
2,--u09WAjW741FdfkJXxNmg,baseline,42.0,0.933364,0.960990,0.089888
3,--u09WAjW741FdfkJXxNmg,recent,30.0,0.972428,0.959711,0.068182
4,-0YrXUvXz8112yHap35V2g,baseline,27.0,0.943068,0.942500,0.150000


In [144]:
category_period_feature_df = (
    category_frequency_df
    .groupby(
        [
            "user_id",
            "period"
        ]
    )
    .apply(
        calculate_category_diversity
    )
    .reset_index()
)

In [145]:
# 6. 기준·최근 기간 분리
baseline_category_df = (
    category_period_feature_df[
        category_period_feature_df[
            "period"
        ] == "baseline"
    ]
    .drop(columns="period")
    .rename(
        columns={
            column: f"baseline_{column}"
            for column
            in category_period_feature_df.columns
            if column not in {
                "user_id",
                "period"
            }
        }
    )
)

In [146]:
recent_category_df = (
    category_period_feature_df[
        category_period_feature_df[
            "period"
        ] == "recent"
    ]
    .drop(columns="period")
    .rename(
        columns={
            column: f"recent_{column}"
            for column
            in category_period_feature_df.columns
            if column not in {
                "user_id",
                "period"
            }
        }
    )
)

In [147]:
# 7. 카테고리 피처 결합
category_feature_df = (
    cohort_df[
        [
            "user_id",
            "churn"
        ]
    ]
    .merge(
        baseline_category_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_category_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

In [148]:
# 8. 다양성 변화 피처
category_feature_df[
    "unique_category_decline_rate"
] = (
    category_feature_df[
        "baseline_unique_category_count"
    ]
    - category_feature_df[
        "recent_unique_category_count"
    ]
) / category_feature_df[
    "baseline_unique_category_count"
]

In [149]:
category_feature_df[
    "category_entropy_decline"
] = (
    category_feature_df[
        "baseline_normalized_category_entropy"
    ]
    - category_feature_df[
        "recent_normalized_category_entropy"
    ]
)

In [150]:
category_feature_df[
    "simpson_diversity_decline"
] = (
    category_feature_df[
        "baseline_simpson_category_diversity"
    ]
    - category_feature_df[
        "recent_simpson_category_diversity"
    ]
)

In [151]:
category_feature_df[
    "top_category_share_increase"
] = (
    category_feature_df[
        "recent_top_category_share"
    ]
    - category_feature_df[
        "baseline_top_category_share"
    ]
)

In [152]:
# 9. 이탈자·유지자 비교
category_comparison_features = [
    "baseline_unique_category_count",
    "recent_unique_category_count",
    "unique_category_decline_rate",
    "baseline_normalized_category_entropy",
    "recent_normalized_category_entropy",
    "category_entropy_decline",
    "baseline_simpson_category_diversity",
    "recent_simpson_category_diversity",
    "simpson_diversity_decline",
    "top_category_share_increase"
]

In [153]:
category_churn_summary_df = (
    category_feature_df
    .groupby("churn")[
        category_comparison_features
    ]
    .agg(
        [
            "mean",
            "median"
        ]
    )
)

category_churn_summary_df.columns = [
    f"{feature}_{stat}"
    for feature, stat
    in category_churn_summary_df.columns
]

category_churn_summary_df = (
    category_churn_summary_df
    .reset_index()
)

category_churn_summary_df


,churn,baseline_unique_category_count_mean,baseline_unique_category_count_median,recent_unique_category_count_mean,recent_unique_category_count_median,unique_category_decline_rate_mean,unique_category_decline_rate_median,baseline_normalized_category_entropy_mean,baseline_normalized_category_entropy_median,recent_normalized_category_entropy_mean,...,category_entropy_decline_mean,category_entropy_decline_median,baseline_simpson_category_diversity_mean,baseline_simpson_category_diversity_median,recent_simpson_category_diversity_mean,recent_simpson_category_diversity_median,simpson_diversity_decline_mean,simpson_diversity_decline_median,top_category_share_increase_mean,top_category_share_increase_median
0,0,41.757117,38.0,33.739855,31.0,0.164521,0.230769,0.940128,0.942591,0.949813,...,-0.009685,-0.012606,0.957528,0.959895,0.936505,0.953488,0.021023,0.005773,0.016749,0.006499
1,1,38.259076,34.0,19.643564,17.0,0.463600,0.519615,0.943854,0.946450,0.954354,...,-0.010501,-0.028433,0.955497,0.957618,0.880418,0.930000,0.075079,0.025045,0.061774,0.024904


In [155]:
# 10. 리뷰 활동량과 상관관계 확인
category_activity_check_df = (
    category_feature_df
    .merge(
        activity_feature_df[
            [
                "user_id",
                "review_count_decline_rate"
            ]
        ],
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

In [156]:
category_correlation_df = (
    category_activity_check_df[
        [
            "review_count_decline_rate",
            "unique_category_decline_rate",
            "category_entropy_decline",
            "simpson_diversity_decline",
            "top_category_share_increase"
        ]
    ]
    .corr()
)

category_correlation_df

,review_count_decline_rate,unique_category_decline_rate,category_entropy_decline,simpson_diversity_decline,top_category_share_increase
review_count_decline_rate,1.000000,0.827433,-0.160894,0.310578,0.274006
unique_category_decline_rate,0.827433,1.000000,-0.077710,0.481136,0.458247
category_entropy_decline,-0.160894,-0.077710,1.000000,0.644231,0.681182
simpson_diversity_decline,0.310578,0.481136,0.644231,1.000000,0.946066
top_category_share_increase,0.274006,0.458247,0.681182,0.946066,1.000000


In [219]:
from pathlib import Path

CATEGORY_FEATURE_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "features"
    / "category_features_v01.parquet"
)

CATEGORY_FEATURE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

assert category_feature_df["user_id"].is_unique
assert len(category_feature_df) == 3_908

category_feature_df.to_parquet(
    CATEGORY_FEATURE_PATH,
    index=False
)

print(CATEGORY_FEATURE_PATH)

C:\Users\playdata2\SKN34-2nd-5Team\data\interim\features\category_features_v01.parquet


In [157]:
# 1. 상관관계 결과 확인
category_correlation_df.round(3)

,review_count_decline_rate,unique_category_decline_rate,category_entropy_decline,simpson_diversity_decline,top_category_share_increase
review_count_decline_rate,1.000,0.827,-0.161,0.311,0.274
unique_category_decline_rate,0.827,1.000,-0.078,0.481,0.458
category_entropy_decline,-0.161,-0.078,1.000,0.644,0.681
simpson_diversity_decline,0.311,0.481,0.644,1.000,0.946
top_category_share_increase,0.274,0.458,0.681,0.946,1.000


In [158]:
print(
    "리뷰 감소율·카테고리 수 감소율:",
    category_correlation_df.loc[
        "review_count_decline_rate",
        "unique_category_decline_rate"
    ]
)

print(
    "리뷰 감소율·엔트로피 감소:",
    category_correlation_df.loc[
        "review_count_decline_rate",
        "category_entropy_decline"
    ]
)

print(
    "리뷰 감소율·Simpson 감소:",
    category_correlation_df.loc[
        "review_count_decline_rate",
        "simpson_diversity_decline"
    ]
)

print(
    "리뷰 감소율·집중도 증가:",
    category_correlation_df.loc[
        "review_count_decline_rate",
        "top_category_share_increase"
    ]
)

리뷰 감소율·카테고리 수 감소율: 0.8274325615286189
리뷰 감소율·엔트로피 감소: -0.16089386461052724
리뷰 감소율·Simpson 감소: 0.3105784583006855
리뷰 감소율·집중도 증가: 0.2740063432366131


In [159]:
# 2. 비슷한 리뷰 수끼리 비교
category_activity_check_df = (
    category_feature_df
    .merge(
        activity_feature_df[
            [
                "user_id",
                "recent_review_count",
                "review_count_decline_rate"
            ]
        ],
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

In [160]:
category_activity_check_df[
    "recent_review_band"
] = pd.cut(
    category_activity_check_df[
        "recent_review_count"
    ],
    bins=[
        0,
        3,
        6,
        10,
        20,
        np.inf
    ],
    labels=[
        "1~3건",
        "4~6건",
        "7~10건",
        "11~20건",
        "21건 이상"
    ],
    include_lowest=True
)

In [161]:
category_band_summary_df = (
    category_activity_check_df
    .groupby(
        [
            "recent_review_band",
            "churn"
        ],
        observed=True
    )
    .agg(
        users=(
            "user_id",
            "count"
        ),
        recent_review_mean=(
            "recent_review_count",
            "mean"
        ),
        recent_category_mean=(
            "recent_unique_category_count",
            "mean"
        ),
        recent_entropy_mean=(
            "recent_normalized_category_entropy",
            "mean"
        ),
        recent_simpson_mean=(
            "recent_simpson_category_diversity",
            "mean"
        ),
        recent_top_share_mean=(
            "recent_top_category_share",
            "mean"
        )
    )
    .reset_index()
)

category_band_summary_df

,recent_review_band,churn,users,recent_review_mean,recent_category_mean,recent_entropy_mean,recent_simpson_mean,recent_top_share_mean
0,1~3건,0,294,2.136054,7.727891,0.955621,0.797274,0.225931
1,1~3건,1,198,1.949495,6.636364,0.937427,0.758373,0.263550
2,4~6건,0,466,4.963519,15.652361,0.976738,0.918491,0.133127
3,4~6건,1,156,4.942308,15.762821,0.975221,0.920276,0.131393
4,7~10건,0,570,8.475439,23.591228,0.964625,0.942101,0.114814
5,7~10건,1,120,8.333333,24.550000,0.966799,0.945690,0.106799
6,11~20건,0,1091,14.831347,34.487626,0.949696,0.955040,0.099688
7,11~20건,1,98,14.071429,34.418367,0.948287,0.954592,0.101072
8,21건 이상,0,881,35.936436,57.627696,0.924193,0.965923,0.086920
9,21건 이상,1,34,28.176471,53.294118,0.930751,0.964096,0.092092


In [162]:
# 맛집 탐방 반경 피처
# 1. 사용자·기간·음식점 중복 제거
spatial_source_df = (
    cohort_review_df[
        [
            "user_id",
            "period",
            "business_id",
            "latitude",
            "longitude"
        ]
    ]
    .drop_duplicates(
        subset=[
            "user_id",
            "period",
            "business_id"
        ]
    )
    .copy()
)

In [165]:
# 2. 사용자별 활동 중심점 계산
spatial_center_df = (
    spatial_source_df
    .groupby(
        [
            "user_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        center_latitude=(
            "latitude",
            "median"
        ),
        center_longitude=(
            "longitude",
            "median"
        )
    )
)

In [166]:
spatial_source_df = (
    spatial_source_df
    .merge(
        spatial_center_df,
        on=[
            "user_id",
            "period"
        ],
        how="left",
        validate="many_to_one"
    )
)

In [167]:
# 3. Haversine 거리 함수
def haversine_distance(
    latitude_1,
    longitude_1,
    latitude_2,
    longitude_2
):
    """
    두 위도·경도 사이의 거리를 km 단위로 계산한다.
    배열 연산이 가능하다.
    """
    earth_radius_km = 6371.0

    lat_1 = np.radians(latitude_1)
    lon_1 = np.radians(longitude_1)
    lat_2 = np.radians(latitude_2)
    lon_2 = np.radians(longitude_2)

    delta_lat = lat_2 - lat_1
    delta_lon = lon_2 - lon_1

    a = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(lat_1)
        * np.cos(lat_2)
        * np.sin(delta_lon / 2) ** 2
    )

    c = 2 * np.arctan2(
        np.sqrt(a),
        np.sqrt(1 - a)
    )

    return earth_radius_km * c

In [168]:
# 4. 활동 중심점으로부터 거리 계산
spatial_source_df[
    "distance_from_center_km"
] = haversine_distance(
    spatial_source_df["latitude"],
    spatial_source_df["longitude"],
    spatial_source_df["center_latitude"],
    spatial_source_df["center_longitude"]
)

In [169]:
spatial_source_df[
    "distance_from_center_km"
].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99
    ]
)

count    140133.000000
mean         48.710591
std         255.781446
min           0.000000
50%           5.379015
90%          22.674664
95%          34.460688
99%        1396.201837
max        3965.980010
Name: distance_from_center_km, dtype: float64

In [170]:
# 5. 사용자·기간별 탐방 반경 집계
spatial_period_df = (
    spatial_source_df
    .groupby(
        [
            "user_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        spatial_business_count=(
            "business_id",
            "nunique"
        ),
        median_radius_km=(
            "distance_from_center_km",
            "median"
        ),
        mean_radius_km=(
            "distance_from_center_km",
            "mean"
        ),
        p90_radius_km=(
            "distance_from_center_km",
            lambda values: values.quantile(0.90)
        ),
        max_radius_km=(
            "distance_from_center_km",
            "max"
        )
    )
)

spatial_period_df.head()

,user_id,period,spatial_business_count,median_radius_km,mean_radius_km,p90_radius_km,max_radius_km
0,--Vu3Gux9nPnLcG9yO_HxA,baseline,18,4.662810,4.997862,9.344368,13.624366
1,--Vu3Gux9nPnLcG9yO_HxA,recent,4,0.718684,1.758646,4.155113,5.403067
2,--u09WAjW741FdfkJXxNmg,baseline,18,3.003629,5.165581,9.455636,25.510828
3,--u09WAjW741FdfkJXxNmg,recent,11,2.368757,4.715140,6.373212,23.313011
4,-0YrXUvXz8112yHap35V2g,baseline,10,8.935847,10.485016,17.107701,32.408668


In [171]:
# 6. 기준·최근 기간 분리
baseline_spatial_df = (
    spatial_period_df[
        spatial_period_df["period"]
        == "baseline"
    ]
    .drop(columns="period")
    .rename(
        columns={
            column: f"baseline_{column}"
            for column
            in spatial_period_df.columns
            if column not in {
                "user_id",
                "period"
            }
        }
    )
)

In [172]:
recent_spatial_df = (
    spatial_period_df[
        spatial_period_df["period"]
        == "recent"
    ]
    .drop(columns="period")
    .rename(
        columns={
            column: f"recent_{column}"
            for column
            in spatial_period_df.columns
            if column not in {
                "user_id",
                "period"
            }
        }
    )
)

In [173]:
# 7. 활동 중심지 이동거리 계산
center_wide_df = (
    spatial_center_df
    .pivot(
        index="user_id",
        columns="period",
        values=[
            "center_latitude",
            "center_longitude"
        ]
    )
)

In [174]:
center_wide_df.columns = [
    f"{period}_{column}"
    for column, period
    in center_wide_df.columns
]

center_wide_df = (
    center_wide_df.reset_index()
)

In [175]:
center_wide_df[
    "center_shift_km"
] = haversine_distance(
    center_wide_df[
        "baseline_center_latitude"
    ],
    center_wide_df[
        "baseline_center_longitude"
    ],
    center_wide_df[
        "recent_center_latitude"
    ],
    center_wide_df[
        "recent_center_longitude"
    ]
)

In [176]:
# 8. 공간 피처 결합
spatial_feature_df = (
    cohort_df[
        [
            "user_id",
            "churn"
        ]
    ]
    .merge(
        baseline_spatial_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_spatial_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        center_wide_df[
            [
                "user_id",
                "center_shift_km"
            ]
        ],
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

In [177]:
# 9. 탐방 반경 변화
spatial_feature_df[
    "p90_radius_decline_km"
] = (
    spatial_feature_df[
        "baseline_p90_radius_km"
    ]
    - spatial_feature_df[
        "recent_p90_radius_km"
    ]
)

In [178]:
spatial_feature_df[
    "median_radius_decline_km"
] = (
    spatial_feature_df[
        "baseline_median_radius_km"
    ]
    - spatial_feature_df[
        "recent_median_radius_km"
    ]
)

In [179]:
spatial_feature_df[
    "recent_spatial_available"
] = (
    spatial_feature_df[
        "recent_spatial_business_count"
    ] >= 2
).astype("int8")

In [181]:
# 10. 이상치 분포 확인
spatial_distribution_df = (
    spatial_feature_df[
        [
            "baseline_p90_radius_km",
            "recent_p90_radius_km",
            "p90_radius_decline_km",
            "center_shift_km"
        ]
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

spatial_distribution_df

,baseline_p90_radius_km,recent_p90_radius_km,p90_radius_decline_km,center_shift_km
count,3908.000000,3908.000000,3908.000000,3908.000000
mean,134.051818,118.794997,15.256821,47.615190
std,412.576121,374.754533,479.147341,257.666099
min,0.494582,0.000000,-3400.141778,0.000000
50%,15.339388,13.654986,0.886159,3.200841
75%,24.569154,23.495052,6.933799,7.297166
90%,287.101404,212.904335,27.369648,15.366369
95%,991.022943,935.855504,740.795889,27.224658
99%,1988.791637,1752.529007,1742.771777,1473.898596
max,3778.633419,3462.576084,3768.434730,3459.724843


In [182]:
# 11. 이탈자·유지자 비교
spatial_comparison_features = [
    "baseline_p90_radius_km",
    "recent_p90_radius_km",
    "p90_radius_decline_km",
    "baseline_median_radius_km",
    "recent_median_radius_km",
    "median_radius_decline_km",
    "center_shift_km",
    "recent_spatial_available"
]

In [183]:
spatial_churn_summary_df = (
    spatial_feature_df
    .groupby("churn")[
        spatial_comparison_features
    ]
    .agg(
        [
            "mean",
            "median"
        ]
    )
)

spatial_churn_summary_df.columns = [
    f"{feature}_{stat}"
    for feature, stat
    in spatial_churn_summary_df.columns
]

spatial_churn_summary_df = (
    spatial_churn_summary_df
    .reset_index()
)

spatial_churn_summary_df

,churn,baseline_p90_radius_km_mean,baseline_p90_radius_km_median,recent_p90_radius_km_mean,recent_p90_radius_km_median,p90_radius_decline_km_mean,p90_radius_decline_km_median,baseline_median_radius_km_mean,baseline_median_radius_km_median,recent_median_radius_km_mean,recent_median_radius_km_median,median_radius_decline_km_mean,median_radius_decline_km_median,center_shift_km_mean,center_shift_km_median,recent_spatial_available_mean,recent_spatial_available_median
0,0,133.952462,15.464891,121.000192,14.289500,12.952270,0.634644,10.453167,5.206484,13.246169,4.953442,-2.793002,0.138541,42.130445,3.095706,0.975469,1.0
1,1,134.593197,14.931944,106.779233,10.314011,27.813964,2.948973,7.902971,4.711507,15.045882,3.350159,-7.142911,0.594715,77.500715,3.989751,0.869637,1.0


In [186]:
# 1. 로그 거리 피처 생성

# 극단값 영향을 줄이기 위해 로그 변환 피처를 추가.
spatial_feature_df[
    "baseline_log_p90_radius"
] = np.log1p(
    spatial_feature_df[
        "baseline_p90_radius_km"
    ]
)

spatial_feature_df[
    "recent_log_p90_radius"
] = np.log1p(
    spatial_feature_df[
        "recent_p90_radius_km"
    ]
)

In [187]:
# 양수일수록 탐방 반경이 축소됨
spatial_feature_df[
    "log_p90_radius_decline"
] = (
    spatial_feature_df[
        "baseline_log_p90_radius"
    ]
    - spatial_feature_df[
        "recent_log_p90_radius"
    ]
)

In [188]:
spatial_feature_df[
    "log_center_shift"
] = np.log1p(
    spatial_feature_df[
        "center_shift_km"
    ]
)

In [189]:
# 2. 음식점 수 구간별 공간 비교
spatial_feature_df[
    "recent_business_band"
] = pd.cut(
    spatial_feature_df[
        "recent_spatial_business_count"
    ],
    bins=[
        0,
        3,
        6,
        10,
        20,
        np.inf
    ],
    labels=[
        "1~3곳",
        "4~6곳",
        "7~10곳",
        "11~20곳",
        "21곳 이상"
    ],
    include_lowest=True
)

In [190]:
spatial_band_summary_df = (
    spatial_feature_df
    .groupby(
        [
            "recent_business_band",
            "churn"
        ],
        observed=True
    )
    .agg(
        users=(
            "user_id",
            "count"
        ),
        recent_business_mean=(
            "recent_spatial_business_count",
            "mean"
        ),
        baseline_p90_median=(
            "baseline_p90_radius_km",
            "median"
        ),
        recent_p90_median=(
            "recent_p90_radius_km",
            "median"
        ),
        p90_decline_median=(
            "p90_radius_decline_km",
            "median"
        ),
        log_p90_decline_median=(
            "log_p90_radius_decline",
            "median"
        ),
        center_shift_median=(
            "center_shift_km",
            "median"
        )
    )
    .reset_index()
)

spatial_band_summary_df

,recent_business_band,churn,users,recent_business_mean,baseline_p90_median,recent_p90_median,p90_decline_median,log_p90_decline_median,center_shift_median
0,1~3곳,0,303,2.151815,15.013084,3.746221,8.848978,1.067166,5.625153
1,1~3곳,1,201,1.955224,14.174308,2.101324,8.916266,1.380660,5.902987
2,4~6곳,0,478,4.951883,15.400043,12.404724,2.734908,0.253641,4.339879
3,4~6곳,1,159,4.943396,15.418070,12.635270,2.016905,0.163135,3.679643
4,7~10곳,0,588,8.469388,14.563015,13.991039,0.513230,0.047447,3.194275
5,7~10곳,1,119,8.386555,15.227826,15.651362,0.178906,0.031006,3.211468
6,11~20곳,0,1079,14.721038,15.098740,14.710002,0.094796,0.001331,2.702455
7,11~20곳,1,95,14.136842,14.129758,13.008486,0.554667,0.053787,2.983038
8,21곳 이상,0,854,35.208431,16.584466,18.950496,-0.684324,-0.046310,2.309018
9,21곳 이상,1,32,28.218750,15.097535,16.292525,-0.347758,-0.001490,2.531332


In [191]:
# 3. 리뷰 감소율과 공간 피처 상관관계
spatial_activity_check_df = (
    spatial_feature_df
    .merge(
        activity_feature_df[
            [
                "user_id",
                "review_count_decline_rate"
            ]
        ],
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

In [192]:
spatial_correlation_df = (
    spatial_activity_check_df[
        [
            "review_count_decline_rate",
            "p90_radius_decline_km",
            "log_p90_radius_decline",
            "center_shift_km",
            "log_center_shift"
        ]
    ]
    .corr()
)

spatial_correlation_df.round(3)

,review_count_decline_rate,p90_radius_decline_km,log_p90_radius_decline,center_shift_km,log_center_shift
review_count_decline_rate,1.000,0.083,0.187,0.052,0.103
p90_radius_decline_km,0.083,1.000,0.809,-0.017,0.033
log_p90_radius_decline,0.187,0.809,1.000,0.019,0.072
center_shift_km,0.052,-0.017,0.019,1.000,0.723
log_center_shift,0.103,0.033,0.072,0.723,1.000


In [193]:
# 4. 장거리 이상치 비율 확인
spatial_outlier_summary_df = (
    spatial_feature_df
    .assign(
        recent_radius_over_100=(
            spatial_feature_df[
                "recent_p90_radius_km"
            ] > 100
        ).astype(int),
        recent_radius_over_500=(
            spatial_feature_df[
                "recent_p90_radius_km"
            ] > 500
        ).astype(int),
        center_shift_over_100=(
            spatial_feature_df[
                "center_shift_km"
            ] > 100
        ).astype(int)
    )
    .groupby("churn")
    .agg(
        users=(
            "user_id",
            "count"
        ),
        radius_over_100_rate=(
            "recent_radius_over_100",
            "mean"
        ),
        radius_over_500_rate=(
            "recent_radius_over_500",
            "mean"
        ),
        center_shift_over_100_rate=(
            "center_shift_over_100",
            "mean"
        )
    )
    .reset_index()
)

rate_columns = [
    "radius_over_100_rate",
    "radius_over_500_rate",
    "center_shift_over_100_rate"
]

spatial_outlier_summary_df[
    rate_columns
] = (
    spatial_outlier_summary_df[
        rate_columns
    ]
    * 100
).round(2)

spatial_outlier_summary_df

,churn,users,radius_over_100_rate,radius_over_500_rate,center_shift_over_100_rate
0,0,3302,10.66,8.00,3.06
1,1,606,10.23,6.77,6.60


In [194]:
# 5. 공간 피처 저장
spatial_feature_df.to_parquet(
    FEATURE_DIR
    / f"spatial_features_{cohort_version}.parquet",
    index=False
)

spatial_band_summary_df.to_csv(
    REPORT_TABLE_DIR
    / f"spatial_band_summary_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

spatial_correlation_df.to_csv(
    REPORT_TABLE_DIR
    / f"spatial_correlation_{cohort_version}.csv",
    encoding="utf-8-sig"
)

spatial_outlier_summary_df.to_csv(
    REPORT_TABLE_DIR
    / f"spatial_outlier_summary_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

In [197]:
# 다음 피처: 평점 성향 변화
# 평점은 리뷰 작성 시점에 확정되므로 데이터 누수 위험이 낮음.
# 1. 평점 분석용 데이터 생성
rating_source_df = (
    cohort_review_df[
        [
            "user_id",
            "period",
            "stars"
        ]
    ]
    .copy()
)

rating_source_df[
    "is_low_rating"
] = (
    rating_source_df["stars"] <= 2
).astype(int)

rating_source_df[
    "is_high_rating"
] = (
    rating_source_df["stars"] >= 4
).astype(int)

In [198]:
# 2. 사용자·기간별 평점 집계
rating_period_df = (
    rating_source_df
    .groupby(
        [
            "user_id",
            "period"
        ],
        as_index=False
    )
    .agg(
        rating_review_count=(
            "stars",
            "count"
        ),
        mean_rating=(
            "stars",
            "mean"
        ),
        median_rating=(
            "stars",
            "median"
        ),
        rating_std=(
            "stars",
            "std"
        ),
        low_rating_rate=(
            "is_low_rating",
            "mean"
        ),
        high_rating_rate=(
            "is_high_rating",
            "mean"
        )
    )
)

rating_period_df.head()

,user_id,period,rating_review_count,mean_rating,median_rating,rating_std,low_rating_rate,high_rating_rate
0,--Vu3Gux9nPnLcG9yO_HxA,baseline,18,4.722222,5.0,0.958280,0.055556,0.944444
1,--Vu3Gux9nPnLcG9yO_HxA,recent,4,4.000000,5.0,2.000000,0.250000,0.750000
2,--u09WAjW741FdfkJXxNmg,baseline,18,4.222222,4.0,0.878204,0.055556,0.833333
3,--u09WAjW741FdfkJXxNmg,recent,11,4.000000,4.0,1.000000,0.090909,0.727273
4,-0YrXUvXz8112yHap35V2g,baseline,10,3.500000,3.5,0.849837,0.100000,0.500000


In [199]:
# 3. 기준·최근 기간 분리
baseline_rating_df = (
    rating_period_df[
        rating_period_df["period"]
        == "baseline"
    ]
    .drop(columns="period")
    .rename(
        columns={
            column: f"baseline_{column}"
            for column
            in rating_period_df.columns
            if column not in {
                "user_id",
                "period"
            }
        }
    )
)

In [200]:
recent_rating_df = (
    rating_period_df[
        rating_period_df["period"]
        == "recent"
    ]
    .drop(columns="period")
    .rename(
        columns={
            column: f"recent_{column}"
            for column
            in rating_period_df.columns
            if column not in {
                "user_id",
                "period"
            }
        }
    )
)

In [201]:
# 4. 평점 피처 결합
rating_feature_df = (
    cohort_df[
        [
            "user_id",
            "churn"
        ]
    ]
    .merge(
        baseline_rating_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        recent_rating_df,
        on="user_id",
        how="left",
        validate="one_to_one"
    )
)

In [202]:
# 5. 평점 변화 피처 생성
rating_feature_df[
    "mean_rating_change"
] = (
    rating_feature_df[
        "recent_mean_rating"
    ]
    - rating_feature_df[
        "baseline_mean_rating"
    ]
)

In [203]:
rating_feature_df[
    "rating_std_change"
] = (
    rating_feature_df[
        "recent_rating_std"
    ]
    - rating_feature_df[
        "baseline_rating_std"
    ]
)

In [204]:
rating_feature_df[
    "low_rating_rate_increase"
] = (
    rating_feature_df[
        "recent_low_rating_rate"
    ]
    - rating_feature_df[
        "baseline_low_rating_rate"
    ]
)

In [205]:
rating_feature_df[
    "high_rating_rate_decline"
] = (
    rating_feature_df[
        "baseline_high_rating_rate"
    ]
    - rating_feature_df[
        "recent_high_rating_rate"
    ]
)

In [206]:
rating_feature_df[
    "recent_rating_std_available"
] = (
    rating_feature_df[
        "recent_rating_review_count"
    ] >= 2
).astype("int8")

In [207]:
# 6. 결측 검증
rating_numeric_columns = (
    rating_feature_df
    .select_dtypes(
        include="number"
    )
    .columns
)

In [208]:
rating_validation_df = pd.DataFrame(
    {
        "feature": rating_numeric_columns,
        "missing_count": [
            rating_feature_df[column]
            .isna()
            .sum()
            for column in rating_numeric_columns
        ],
        "infinite_count": [
            np.isinf(
                rating_feature_df[column]
            ).sum()
            for column in rating_numeric_columns
        ]
    }
)

rating_validation_df

,feature,missing_count,infinite_count
0,churn,0,0
1,baseline_rating_review_count,0,0
2,baseline_mean_rating,0,0
3,baseline_median_rating,0,0
4,baseline_rating_std,0,0
5,baseline_low_rating_rate,0,0
6,baseline_high_rating_rate,0,0
7,recent_rating_review_count,0,0
8,recent_mean_rating,0,0
9,recent_median_rating,0,0


In [209]:
# 7. 이탈자·유지자 비교
rating_comparison_features = [
    "baseline_mean_rating",
    "recent_mean_rating",
    "mean_rating_change",
    "baseline_rating_std",
    "recent_rating_std",
    "rating_std_change",
    "baseline_low_rating_rate",
    "recent_low_rating_rate",
    "low_rating_rate_increase",
    "high_rating_rate_decline"
]

In [210]:
rating_churn_summary_df = (
    rating_feature_df
    .groupby("churn")[
        rating_comparison_features
    ]
    .agg(
        [
            "mean",
            "median"
        ]
    )
)

rating_churn_summary_df.columns = [
    f"{feature}_{stat}"
    for feature, stat
    in rating_churn_summary_df.columns
]

rating_churn_summary_df = (
    rating_churn_summary_df
    .reset_index()
)

rating_churn_summary_df

,churn,baseline_mean_rating_mean,baseline_mean_rating_median,recent_mean_rating_mean,recent_mean_rating_median,mean_rating_change_mean,mean_rating_change_median,baseline_rating_std_mean,baseline_rating_std_median,recent_rating_std_mean,...,rating_std_change_mean,rating_std_change_median,baseline_low_rating_rate_mean,baseline_low_rating_rate_median,recent_low_rating_rate_mean,recent_low_rating_rate_median,low_rating_rate_increase_mean,low_rating_rate_increase_median,high_rating_rate_decline_mean,high_rating_rate_decline_median
0,0,3.862241,3.904762,3.877707,3.980655,0.015466,0.046581,1.001052,0.976284,0.979483,...,-0.019428,-0.013597,0.132631,0.090909,0.138697,0.080000,0.006066,0.0,-0.003743,-0.011713
1,1,3.929909,3.967134,3.856457,4.000000,-0.073452,0.023253,0.980525,0.954499,0.968428,...,-0.004986,-0.036937,0.123635,0.083333,0.174718,0.035099,0.051083,0.0,0.020290,-0.017621


In [211]:
# 8. 결과 저장
rating_feature_df.to_parquet(
    FEATURE_DIR
    / f"rating_features_{cohort_version}.parquet",
    index=False
)

rating_validation_df.to_csv(
    REPORT_TABLE_DIR
    / f"rating_feature_validation_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

rating_churn_summary_df.to_csv(
    REPORT_TABLE_DIR
    / f"rating_churn_summary_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("평점 성향 피처 저장 완료")

평점 성향 피처 저장 완료


In [213]:
# # 마지막 검증: 리뷰 반응
# useful, cool, funny는 계산 자체는 가능하지만 모델 피처로 사용하면 안 될 가능성이 높아.

# 예를 들어 2017년 리뷰의 useful=10이:

# 2017년에 받은 반응인지
# 2018년에 받은 반응인지
# 2019년 이후 받은 반응인지

# 구분할 수 없어. 데이터 수집 시점까지 누적된 값이면 2019년 이후의 정보가 포함되어 미래 정보 누수가 발생한다.

# 따라서 반응 피처는 분포만 확인하고 메인 모델에서는 제외하자.
# 1. 반응 데이터 분포 확인
reaction_columns = [
    "useful",
    "cool",
    "funny"
]

In [214]:
reaction_audit_results = []

for column in reaction_columns:
    reaction_audit_results.append(
        {
            "feature": column,
            "total_rows": len(
                cohort_review_df
            ),
            "missing_count": (
                cohort_review_df[
                    column
                ].isna().sum()
            ),
            "zero_count": (
                cohort_review_df[
                    column
                ] == 0
            ).sum(),
            "zero_rate_pct": round(
                (
                    cohort_review_df[
                        column
                    ] == 0
                ).mean()
                * 100,
                2
            ),
            "mean": (
                cohort_review_df[
                    column
                ].mean()
            ),
            "median": (
                cohort_review_df[
                    column
                ].median()
            ),
            "max": (
                cohort_review_df[
                    column
                ].max()
            )
        }
    )

reaction_audit_df = pd.DataFrame(
    reaction_audit_results
)

reaction_audit_df

,feature,total_rows,missing_count,zero_count,zero_rate_pct,mean,median,max
0,useful,144482,0,52659,36.45,2.234465,1.0,182
1,cool,144482,0,79276,54.87,1.383418,0.0,174
2,funny,144482,0,107465,74.38,0.660408,0.0,101


In [215]:
# 2. 결과 저장
reaction_audit_df.to_csv(
    REPORT_TABLE_DIR
    / f"reaction_data_audit_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("리뷰 반응 데이터 검증 완료")

리뷰 반응 데이터 검증 완료


In [216]:
feature_decision_data = [
    {
        "feature_group": "리뷰 활동량",
        "feature": "review_count_decline_rate",
        "decision": "채택",
        "priority": "P0",
        "reason": "이탈자 리뷰 감소율이 크게 나타남"
    },
    {
        "feature_group": "리뷰 활동량",
        "feature": "active_month_decline_rate",
        "decision": "채택",
        "priority": "P0",
        "reason": "이탈자 활동 월수가 크게 감소"
    },
    {
        "feature_group": "리뷰 활동량",
        "feature": "reviews_per_active_month_decline_rate",
        "decision": "채택 후보",
        "priority": "P1",
        "reason": "활동한 월의 리뷰 밀도 감소 반영"
    },
    {
        "feature_group": "작성 간격",
        "feature": "recent_mean_interval_days",
        "decision": "채택",
        "priority": "P0",
        "reason": "이탈자의 최근 작성 간격 증가"
    },
    {
        "feature_group": "작성 간격",
        "feature": "mean_interval_increase_days",
        "decision": "채택",
        "priority": "P0",
        "reason": "기준 기간 대비 작성 간격 증가 반영"
    },
    {
        "feature_group": "작성 간격",
        "feature": "recent_recency_days",
        "decision": "채택",
        "priority": "P0",
        "reason": "이탈자의 마지막 리뷰 이후 공백이 큼"
    },
    {
        "feature_group": "작성 간격",
        "feature": "recent_interval_available",
        "decision": "채택",
        "priority": "P0",
        "reason": "최근 리뷰 1건 사용자의 결측 상태 보존"
    },
    {
        "feature_group": "신규 음식점",
        "feature": "new_business_rate_decline",
        "decision": "제외",
        "priority": "-",
        "reason": "중앙값 0이며 이탈 집단 차이가 작음"
    },
    {
        "feature_group": "음식점 범위",
        "feature": "unique_business_decline_rate",
        "decision": "설명용",
        "priority": "P2",
        "reason": "리뷰 감소율과 상관관계가 매우 높음"
    },
    {
        "feature_group": "카테고리",
        "feature": "unique_category_decline_rate",
        "decision": "설명용",
        "priority": "P2",
        "reason": "리뷰 수 감소의 영향을 크게 받음"
    },
    {
        "feature_group": "카테고리",
        "feature": "category_entropy_decline",
        "decision": "제외",
        "priority": "-",
        "reason": "집단 차이가 거의 없음"
    },
    {
        "feature_group": "카테고리",
        "feature": "simpson_diversity_decline",
        "decision": "제외",
        "priority": "-",
        "reason": "리뷰 수 구간 통제 후 차이가 사라짐"
    },
    {
        "feature_group": "탐방 반경",
        "feature": "log_p90_radius_decline",
        "decision": "실험용",
        "priority": "P1",
        "reason": "전체 차이는 있으나 음식점 수 구간별 일관성 부족"
    },
    {
        "feature_group": "활동 중심지",
        "feature": "log_center_shift",
        "decision": "보조 후보",
        "priority": "P1",
        "reason": "장거리 중심지 이동 비율이 이탈자에게 높음"
    },
    {
        "feature_group": "평점",
        "feature": "mean_rating_change",
        "decision": "보조 후보",
        "priority": "P1",
        "reason": "이탈자 평균 평점이 소폭 하락"
    },
    {
        "feature_group": "평점",
        "feature": "low_rating_rate_increase",
        "decision": "보조 후보",
        "priority": "P1",
        "reason": "이탈자의 저평점 비율 평균 증가"
    },
    {
        "feature_group": "리뷰 반응",
        "feature": "useful/cool/funny",
        "decision": "제외",
        "priority": "-",
        "reason": "반응 발생 시점이 없어 미래 정보 누수 가능"
    }
]

feature_decision_df = pd.DataFrame(
    feature_decision_data
)

feature_decision_df

,feature_group,feature,decision,priority,reason
0,리뷰 활동량,review_count_decline_rate,채택,P0,이탈자 리뷰 감소율이 크게 나타남
1,리뷰 활동량,active_month_decline_rate,채택,P0,이탈자 활동 월수가 크게 감소
2,리뷰 활동량,reviews_per_active_month_decline_rate,채택 후보,P1,활동한 월의 리뷰 밀도 감소 반영
3,작성 간격,recent_mean_interval_days,채택,P0,이탈자의 최근 작성 간격 증가
4,작성 간격,mean_interval_increase_days,채택,P0,기준 기간 대비 작성 간격 증가 반영
5,작성 간격,recent_recency_days,채택,P0,이탈자의 마지막 리뷰 이후 공백이 큼
6,작성 간격,recent_interval_available,채택,P0,최근 리뷰 1건 사용자의 결측 상태 보존
7,신규 음식점,new_business_rate_decline,제외,-,중앙값 0이며 이탈 집단 차이가 작음
8,음식점 범위,unique_business_decline_rate,설명용,P2,리뷰 감소율과 상관관계가 매우 높음
9,카테고리,unique_category_decline_rate,설명용,P2,리뷰 수 감소의 영향을 크게 받음


In [217]:
feature_decision_df.to_csv(
    REPORT_TABLE_DIR
    / f"feature_feasibility_decisions_{cohort_version}.csv",
    index=False,
    encoding="utf-8-sig"
)

print("최종 피처 판정표 저장 완료")

최종 피처 판정표 저장 완료


피처 검증 최종 결론

강하게 확인된 이탈 징후는 다음이야.

리뷰 작성량 감소
활동 월수 감소
리뷰 작성 간격 증가
마지막 리뷰 이후 공백 증가

부분적으로 확인된 신호는 다음이야.

일부 평점 성향 변화
활동 중심지의 큰 이동
탐방 반경 축소

독립적인 신호로 확인되지 않은 가설은 다음이야.

신규 음식점 탐색 비율 감소
카테고리 구성 다양성 감소
동일 음식점 재리뷰 감소